## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## Data Preprocessing

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
print(df.shape)
df.head()

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Fill missing values
df["Age"].fillna(df["Age"].median(), inplace=True)
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)
df["Fare"].fillna(df["Fare"].median(), inplace=True)

# Encode categoricals
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

print("Features:", df.columns.tolist())
df.info()

In [ ]:
X = df.drop("Survived", axis=1).values
y = df["Survived"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Model 1 – Basic Neural Network

In [ ]:
def build_basic_model(input_dim, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1,  activation="sigmoid")
    ], name="BasicNN")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

basic_model = build_basic_model(X_train.shape[1])
basic_model.summary()

In [ ]:
history_basic = basic_model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=80,
    batch_size=32,
    verbose=0
)
print("Basic NN trained ✔")

## Model 2 – Deep Neural Network

In [ ]:
def build_deep_model(input_dim, learning_rate=0.001, l2_lambda=0.01, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation="relu"),
        layers.Dense(1,  activation="sigmoid")
    ], name="DeepNN")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

deep_model = build_deep_model(X_train.shape[1])
deep_model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history_deep = deep_model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)
print(f"Deep NN trained ✔  (stopped at epoch {len(history_deep.history['loss'])})")

## Training Errors

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # Loss
    axes[0].plot(history.history["loss"],     label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss", linestyle="--")
    axes[0].set_title("Loss over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary Cross-Entropy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(history.history["accuracy"],     label="Train Accuracy")
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy", linestyle="--")
    axes[1].set_title("Accuracy over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_basic, "Basic NN – Training History")
plot_history(history_deep,  "Deep NN  – Training History")

## Confusion Matrix

In [ ]:
y_pred_basic = (basic_model.predict(X_test) >= 0.5).astype(int).flatten()
y_pred_deep  = (deep_model.predict(X_test)  >= 0.5).astype(int).flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_basic, y_pred_deep],
    ["Basic NN", "Deep NN"]
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Survived", "Survived"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix – {title}", fontweight="bold")

plt.tight_layout()
plt.show()

print("=== Basic NN ===")
print(classification_report(y_test, y_pred_basic, target_names=["Not Survived", "Survived"]))

print("=== Deep NN ===")
print(classification_report(y_test, y_pred_deep,  target_names=["Not Survived", "Survived"]))

## ROC / AUC

In [ ]:
y_prob_basic = basic_model.predict(X_test).flatten()
y_prob_deep  = deep_model.predict(X_test).flatten()

fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_basic)
fpr_d, tpr_d, _ = roc_curve(y_test, y_prob_deep)

auc_basic = roc_auc_score(y_test, y_prob_basic)
auc_deep  = roc_auc_score(y_test, y_prob_deep)

plt.figure(figsize=(8, 6))
plt.plot(fpr_b, tpr_b, label=f"Basic NN  (AUC = {auc_basic:.3f})", linewidth=2)
plt.plot(fpr_d, tpr_d, label=f"Deep NN   (AUC = {auc_deep:.3f})",  linewidth=2, linestyle="--")
plt.plot([0, 1], [0, 1], "k:", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Basic NN vs Deep NN", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Basic NN AUC: {auc_basic:.4f}")
print(f"Deep  NN AUC: {auc_deep:.4f}")

## Hyperparameter Experiments

In [ ]:
hyperparams = [
    {"lr": 0.01,   "batch": 16,  "epochs": 100, "l2": 0.001, "dropout": 0.2},
    {"lr": 0.001,  "batch": 32,  "epochs": 150, "l2": 0.01,  "dropout": 0.3},
    {"lr": 0.0001, "batch": 64,  "epochs": 200, "l2": 0.001, "dropout": 0.4},
    {"lr": 0.005,  "batch": 32,  "epochs": 120, "l2": 0.05,  "dropout": 0.3},
]

results = []

for cfg in hyperparams:
    m = build_deep_model(
        X_train.shape[1],
        learning_rate=cfg["lr"],
        l2_lambda=cfg["l2"],
        dropout_rate=cfg["dropout"]
    )
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    h = m.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=cfg["epochs"],
        batch_size=cfg["batch"],
        callbacks=[es],
        verbose=0
    )
    y_prob = m.predict(X_test).flatten()
    auc    = roc_auc_score(y_test, y_prob)
    acc    = np.mean((y_prob >= 0.5).astype(int) == y_test)

    results.append({
        "LR": cfg["lr"], "Batch": cfg["batch"],
        "L2": cfg["l2"], "Dropout": cfg["dropout"],
        "Epochs run": len(h.history["loss"]),
        "Test Acc": round(acc, 4), "AUC": round(auc, 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [f"LR={r['LR']}\nBatch={r['Batch']}\nL2={r['L2']}" for _, r in results_df.iterrows()]

axes[0].bar(labels, results_df["Test Acc"], color="steelblue", edgecolor="black")
axes[0].set_title("Test Accuracy by Hyperparameter Config", fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.7, 1.0)
axes[0].tick_params(axis='x', labelsize=8)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(labels, results_df["AUC"], color="coral", edgecolor="black")
axes[1].set_title("AUC by Hyperparameter Config", fontweight="bold")
axes[1].set_ylabel("AUC")
axes[1].set_ylim(0.7, 1.0)
axes[1].tick_params(axis='x', labelsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Discussion

- The basic model (1 hidden layer) is a good baseline but has limited capacity.
- The deep model benefits from Dropout and L2 regularization to avoid overfitting on a small dataset like Titanic.
- A high learning rate can cause divergence; too low slows convergence — `0.001` works well with Adam.
- Smaller batch sizes add noise during training which acts as an implicit regularizer.
- AUC is more informative than accuracy alone when classes are slightly imbalanced.
- On tabular datasets this size, a well-regularized shallow model often matches a deep one.